Enrolment datasets viaulization

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import pandas as pd
import os

folder_path = "/content/drive/MyDrive/uidai/api_data_aadhar_enrolment/api_data_aadhar_enrolment"

# List all CSV files
csv_files = [f for f in os.listdir(folder_path) if f.endswith(".csv")]

print("CSV files found:", csv_files)

# Read and merge
df_list = []
for file in csv_files:
    file_path = os.path.join(folder_path, file)
    temp_df = pd.read_csv(file_path)
    df_list.append(temp_df)

# Merge all CSVs
df = pd.concat(df_list, ignore_index=True)

df.head()


In [ ]:
print("Total rows:", df.shape[0])
print("Total columns:", df.shape[1])


In [ ]:
for file in csv_files:
    print(file, pd.read_csv(os.path.join(folder_path, file)).columns.tolist())


In [ ]:
output_path = "/content/drive/MyDrive/uidai/api_data_aadhar_enrolment/merged_aadhar_enrolment.csv"
df.to_csv(output_path, index=False)

print("Merged file saved to:", output_path)


In [ ]:
df.groupby("state")[["age_0_5","age_5_17","age_18_greater"]].sum().plot(kind="bar")


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)


In [ ]:
age_cols = ["age_0_5", "age_5_17", "age_18_greater"]
df[age_cols] = df[age_cols].apply(pd.to_numeric, errors="coerce")

df = df.dropna(subset=age_cols)


In [ ]:
state_age = (
    df.groupby("state")[age_cols]
    .sum()
    .sort_values(by="age_5_17", ascending=False)
)

ax = state_age.plot(kind="bar")
plt.title("Age Group Distribution by State")
plt.xlabel("State")
plt.ylabel("Population Count")
plt.legend(title="Age Group")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()

plt.savefig(
    "/content/drive/MyDrive/uidai/api_data_aadhar_enrolment/api_data_aadhar_enrolment/state_age_distribution.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()


In [ ]:
district_age = (
    df.groupby("district")[age_cols]
    .sum()
    .sort_values(by="age_5_17", ascending=False)
    .head(15)   # limit for readability
)

district_age.plot(kind="bar", stacked=True)
plt.title("District-wise Age Composition (Top 15)")
plt.xlabel("District")
plt.ylabel("Population Count")
plt.xticks(rotation=75, ha="right")
plt.tight_layout()

plt.savefig(
    "/content/drive/MyDrive/uidai/api_data_aadhar_enrolment/api_data_aadhar_enrolment/district_age_stacked.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()


In [ ]:
top_school = (
    df.groupby("district")["age_5_17"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

sns.barplot(
    x=top_school.values,
    y=top_school.index,
    palette="viridis"
)

plt.title("Top 10 Districts by School-age Population (5–17)")
plt.xlabel("Population (Age 5–17)")
plt.ylabel("District")
plt.tight_layout()

plt.savefig(
    "/content/drive/MyDrive/uidai/api_data_aadhar_enrolment/api_data_aadhar_enrolment/top_school_age_districts.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()


In [ ]:
total_age = df[age_cols].sum()

plt.figure(figsize=(7,7))
plt.pie(
    total_age,
    labels=["Age 0–5", "Age 5–17", "Age 18+"],
    autopct="%1.1f%%",
    startangle=140,
    explode=(0.05, 0.05, 0)
)

plt.title("Overall Age Group Distribution")

plt.savefig(
    "/content/drive/MyDrive/uidai/api_data_aadhar_enrolment/api_data_aadhar_enrolment/overall_age_share.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()


 Intreactive charts


In [ ]:
!pip install plotly
import plotly.express as px
import plotly.graph_objects as go


In [ ]:
state_age = df.groupby("state")[["age_0_5","age_5_17","age_18_greater"]].sum().reset_index()

fig = px.bar(
    state_age,
    x="state",
    y=["age_0_5","age_5_17","age_18_greater"],
    title="Interactive Age Group Distribution by State",
    labels={"value":"Population Count", "state":"State"},
    barmode="group"
)

fig.show()


In [ ]:
district_age = (
    df.groupby("district")[["age_0_5","age_5_17","age_18_greater"]]
    .sum()
    .reset_index()
    .sort_values(by="age_5_17", ascending=False)
    .head(15)
)

fig = px.bar(
    district_age,
    x="district",
    y=["age_0_5","age_5_17","age_18_greater"],
    title="Top 15 Districts – Age Composition",
    barmode="stack"
)

fig.update_layout(xaxis_tickangle=-45)
fig.show()


In [ ]:
total_age = df[["age_0_5","age_5_17","age_18_greater"]].sum().reset_index()
total_age.columns = ["Age Group", "Population"]

fig = px.pie(
    total_age,
    names="Age Group",
    values="Population",
    title="Overall Age Group Distribution"
)

fig.show()


New

In [ ]:
df["date"] = pd.to_datetime(df["date"], errors="coerce")


In [ ]:
date_trend = df.groupby("date")[["age_0_5","age_5_17","age_18_greater"]].sum().reset_index()

fig = px.line(
    date_trend,
    x="date",
    y=["age_0_5","age_5_17","age_18_greater"],
    markers=True,
    title="Date-wise UIDAI Age Group Trend"
)

fig.show()


In [ ]:
fig = px.line(
    df,
    x="date",
    y="age_5_17",
    color="state",
    title="State-wise Trend: School-age Population (5–17)"
)

fig.show()


📘 Project Title

UIDAI Aadhaar Enrolment Data Analysis for Demographic Decision-Making

🎯 Objective

The objective of this project is to analyze UIDAI Aadhaar enrolment data across Indian states and districts to identify demographic trends and support data-driven decision-making in education, healthcare, and welfare planning.

🗂️ Data Description

The dataset consists of Aadhaar enrolment records with the following attributes:

Date of record

State and District

Pincode

Population counts across three age groups:

0–5 years

5–17 years

18 years and above

Multiple CSV files were collected via UIDAI APIs and merged into a single dataset using Python.

🛠️ Tools & Technologies

Google Colab

Python

Pandas – data processing

Matplotlib & Seaborn – static visualization

Plotly – interactive dashboards

📊 Methodology

Mounted Google Drive and loaded multiple CSV files

Merged datasets using Pandas

Cleaned and standardized age-group data

Performed aggregations at state and district levels

Visualized data using static and interactive charts

Conducted date-wise trend analysis

📈 Key Findings

Bihar and Uttar Pradesh show significantly higher school-age (5–17) populations

Several districts have high dependency ratios, indicating pressure on working-age adults

Urban regions like Bengaluru Urban show balanced age distributions

Date-wise trends indicate consistent growth in child and adolescent enrolments

🧠 Decision-Making Insights

High-priority states require increased education and child welfare investment

District-level targeting is essential for efficient policy implementation

Trend analysis supports future forecasting for infrastructure planning

✅ Conclusion

This analysis demonstrates how UIDAI Aadhaar enrolment data can be transformed into actionable insights through data visualization and trend analysis. Interactive dashboards further enhance policy-level decision-making by enabling real-time exploration of demographic patterns.

🔮 Future Scope

Predictive modeling for enrolment growth

Integration with health and education datasets

GIS-based spatial analysis

-------------------------Try new things----------------------

In [ ]:
!pip install plotly

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots



In [ ]:
# Ensure correct data types
df["date"] = pd.to_datetime(df["date"], errors="coerce")

age_cols = ["age_0_5", "age_5_17", "age_18_greater"]
df[age_cols] = df[age_cols].apply(pd.to_numeric, errors="coerce")

df = df.dropna(subset=age_cols)


In [ ]:
total_records = len(df)
total_population = df[age_cols].sum().sum()
total_children = df["age_0_5"].sum()
total_school_age = df["age_5_17"].sum()
total_adults = df["age_18_greater"].sum()


In [ ]:
fig = make_subplots(
    rows=3, cols=2,
    specs=[
        [{"type": "indicator"}, {"type": "indicator"}],
        [{"type": "xy"}, {"type": "xy"}],
        [{"type": "xy"}, {"type": "domain"}]   # ✅ Pie needs domain
    ],
    subplot_titles=(
        "Total Population",
        "School-age Population (5–17)",
        "State-wise Age Distribution",
        "Top 10 Districts – School-age Population",
        "Date-wise Age Trend",
        "Overall Age Group Share"
    )
)


In [ ]:
fig.add_trace(
    go.Indicator(
        mode="number",
        value=total_population,
        number={"valueformat": ","},
        title={"text": "Total Population"}
    ),
    row=1, col=1
)

fig.add_trace(
    go.Indicator(
        mode="number",
        value=total_school_age,
        number={"valueformat": ","},
        title={"text": "Age 5–17 Population"}
    ),
    row=1, col=2
)


In [ ]:
state_age = df.groupby("state")[age_cols].sum().reset_index()

for col in age_cols:
    fig.add_trace(
        go.Bar(
            x=state_age["state"],
            y=state_age[col],
            name=col.replace("_", " ").title()
        ),
        row=2, col=1
    )


In [ ]:
top_districts = (
    df.groupby("district")["age_5_17"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)

fig.add_trace(
    go.Bar(
        x=top_districts["age_5_17"],
        y=top_districts["district"],
        orientation="h",
        name="Age 5–17",
        marker_color="teal"
    ),
    row=2, col=2
)


In [ ]:
total_age = df[age_cols].sum()

fig.add_trace(
    go.Pie(
        labels=["Age 0–5", "Age 5–17", "Age 18+"],
        values=total_age.values,
        hole=0.4
    ),
    row=3, col=2
)


In [ ]:
date_trend = df.groupby("date")[age_cols].sum().reset_index()

for col in age_cols:
    fig.add_trace(
        go.Scatter(
            x=date_trend["date"],
            y=date_trend[col],
            mode="lines+markers",
            name=col.replace("_", " ").title()
        ),
        row=3, col=1
    )


In [ ]:
total_age = df[age_cols].sum()

fig.add_trace(
    go.Pie(
        labels=["Age 0–5", "Age 5–17", "Age 18+"],
        values=total_age.values,
        hole=0.4
    ),
    row=3, col=2
)


In [ ]:
fig.update_layout(
    height=1200,
    title_text="UIDAI Aadhaar Enrolment – Demographic Decision Dashboard",
    template="plotly_white",
    showlegend=True
)

fig.show()
